# Pipelines de Scikit-Learn para Clustering

### Dataset: Digits

Usaremos el **dataset Digits** de scikit-learn, que contiene imágenes de dígitos escritos a mano (0-9). Cada imagen es de 8×8 píxeles, representada como 64 features (intensidades de píxeles).

**El reto**: ¿Puede K-Means descubrir las 10 clases de dígitos sin ver nunca las etiquetas?

## 1. Importar las Bibliotecas Necesarias

Necesitaremos herramientas para cargar datos, preprocesamiento, clustering, evaluación y visualización.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

# Importaciones de Scikit-learn
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

# Métricas de evaluación de clustering
from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    homogeneity_score,
    completeness_score,
    v_measure_score
)

# Configuración de visualización
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

## 2. Cargar y Explorar el Dataset Digits

El **dataset Digits** contiene 1,797 imágenes de dígitos escritos a mano (0-9), cada una representada como una cuadrícula de píxeles de 8×8 (64 features en total). Cada valor de píxel varía de 0 (blanco) a 16 (negro).

### Características del Dataset

- **Muestras**: 1,797 imágenes de dígitos escritos a mano
- **Features**: 64 (intensidades de píxeles de 8×8)
- **Clases**: 10 (dígitos 0-9)
- **Rango de features**: 0-16 (intensidad de escala de grises)

**Importante**: Aunque este dataset tiene etiquetas, las usaremos SOLO para evaluación. El algoritmo de clustering no verá las etiquetas durante el entrenamiento (¡eso es lo que lo hace **no supervisado**!).

In [ ]:
digits = load_digits() # Cargar el dataset Digits
X = digits.data  # 64 features (píxeles 8x8)
y_true = digits.target  # Etiquetas reales (¡solo para evaluación!)

print(f"Forma del dataset: {X.shape[0]:,} imágenes, {X.shape[1]} features (píxeles 8×8)")
print(f"Clases: {len(np.unique(y_true))} dígitos (0-9)")
print(f"Rango de features: [{X.min():.0f}, {X.max():.0f}] (intensidad de píxel)")

# Visualizar dígitos de muestra
fig, axes = plt.subplots(4, 10, figsize=(15, 6))
fig.suptitle('Muestra de Dígitos Escritos a Mano del Dataset', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < len(X):
        # Reformar 64 features de vuelta a imagen 8x8
        image = X[i].reshape(8, 8)
        ax.imshow(image, cmap='gray')
        ax.set_title(f'Etiqueta: {y_true[i]}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Mostrar la distribución de muestras entre las 10 clases de dígitos
unique_classes, class_counts = np.unique(y_true, return_counts=True)

print("Distribución de muestras por clase de dígito:")
print(f"{'Dígito':<8} {'Cantidad':>8} {'Porcentaje':>12}")
print("-" * 30)

for digit, count in zip(unique_classes, class_counts):
    percentage = (count / len(y_true)) * 100
    bar = '█' * (count // 10)
    print(f"{digit:<8} {count:>8} ({percentage:>6.2f}%)  {bar}")

print("-" * 30)
print(f"{'Total':<8} {len(y_true):>8} ({100.0:>6.2f}%)")

### Demostración del Escalado de Features

Antes de proceder con el clustering, analizamos el efecto del escalado de features. K-Means se basa en cálculos de distancia (distancia euclídea), por lo que es fundamental que todas las features contribuyan por igual.

En la siguiente celda:
1. Seleccionamos algunas features de ejemplo (píxeles).
2. Calculamos y mostramos sus estadísticas (Media, Desv. Estándar, Mín, Máx) **antes del escalado**, observando que los valores de píxel crudos pueden variar significativamente.
3. Aplicamos `StandardScaler` para transformar los datos.
4. Calculamos las estadísticas **después del escalado**, confirmando que todas las features ahora tienen media $\approx$ 0 y desviación estándar $\approx$ 1.
5. Visualizamos las distribuciones usando boxplots para verificar la normalización.

In [ ]:
# Comparar algunas features antes y después del escalado
selected_features = [0, 20, 40, 60]
feature_names = [f'Píxel {i}' for i in selected_features]

# Estadísticas originales
print("\nANTES DEL ESCALADO (Features Originales):")
print(f"{'Feature':<12} {'Media':>8} {'Desv.Est.':>8} {'Mín':>8} {'Máx':>8}")
print("-" * 48)
for idx, name in zip(selected_features, feature_names):
    col = X[:, idx]
    print(f"{name:<12} {col.mean():8.2f} {col.std():8.2f} {col.min():8.2f} {col.max():8.2f}")

# Aplicar escalado
scaler_demo = StandardScaler()
X_scaled_demo = scaler_demo.fit_transform(X)

# Estadísticas escaladas
print("\nDESPUÉS DEL ESCALADO (Features Estandarizadas):")
print(f"{'Feature':<12} {'Media':>8} {'Desv.Est.':>8} {'Mín':>8} {'Máx':>8}")
print("-" * 48)
for idx, name in zip(selected_features, feature_names):
    col = X_scaled_demo[:, idx]
    print(f"{name:<12} {col.mean():8.5f} {col.std():8.5f} {col.min():8.2f} {col.max():8.2f}")

print("\n✓ Todas las features ahora tienen media ≈ 0 y desv. estándar ≈ 1")
print("✓ Todas las features contribuyen por igual a los cálculos de distancia")

# Visualizar el efecto del escalado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Antes del escalado
axes[0].boxplot([X[:, idx] for idx in selected_features], tick_labels=feature_names)
axes[0].set_title('Antes del Escalado', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Intensidad de Píxel')
axes[0].grid(True, alpha=0.3)

# Después del escalado
axes[1].boxplot([X_scaled_demo[:, idx] for idx in selected_features], tick_labels=feature_names)
axes[1].set_title('Después del Escalado (Estandarizado)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Valor Estandarizado')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='r', linestyle='--', linewidth=1, alpha=0.5)

plt.tight_layout()
plt.show()

### Interpretación de los Resultados

**1. Salida de Estadísticas:**
- **Antes del Escalado**: Se puede ver que diferentes píxeles tienen estadísticas muy distintas (algunos pueden variar entre 0-255, otros pueden ser principalmente oscuros). Si usáramos los datos crudos, los cálculos de distancia estarían sesgados hacia las features con rangos numéricos más grandes.
- **Después del Escalado**: Todas las features seleccionadas ahora tienen una Media muy cercana a $0$ y una Desviación Estándar de $1$. Esto garantiza que cada píxel contribuya por igual a las medidas de similitud.

**2. Visualización con Boxplot:**
El gráfico anterior visualiza la distribución de valores de los píxeles seleccionados.

- La **Caja** representa el Rango Intercuartílico (IQR), que contiene el 50% central de los datos (desde el percentil 25 hasta el 75).
- La **Línea** dentro de la caja es la Mediana (percentil 50).
- Los **Bigotes** se extienden hasta los puntos de datos más extremos que no se consideran atípicos (típicamente $1.5 \times IQR$).
- Los **Valores Atípicos** (puntos individuales) son valores que caen más allá de los bigotes.

Observa cómo en el gráfico *Después del Escalado*, las cajas están alineadas alrededor de la línea roja discontinua ($y=0$) y tienen tamaños similares. Esto confirma que los datos están esencialmente centrados y escalados, listos para el clustering.

## 4. Construir y Entrenar el Pipeline de Clustering

Ahora crearemos un pipeline que combina preprocesamiento y clustering. Usaremos K=10 clusters ya que sabemos que hay 10 dígitos (¡aunque el algoritmo no lo sabe!).

### Definición y Entrenamiento del Pipeline

1. **StandardScaler**: Asegura que todas las features contribuyan por igual
2. **KMeans**: Encuentra K centros de clusters que minimizan la varianza dentro del cluster

Parámetros de K-Means:
- `n_clusters=10`: Número de clusters a encontrar
- `random_state=42`: Para reproducibilidad (K-Means tiene inicialización aleatoria)
- `n_init=10`: Número de veces que se ejecuta K-Means con diferentes inicializaciones (conserva el mejor resultado)
- `max_iter=300`: Máximo de iteraciones antes de detenerse

**Importante**: Elegimos K=10 porque sabemos que hay 10 dígitos. En escenarios realmente no supervisados, elegir K es un reto que abordaremos más adelante.

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('kmeans', KMeans(
        n_clusters=10,       # Queremos 10 clusters (uno por dígito)
        random_state=42,     # Para reproducibilidad
        n_init=10,           # Ejecutar 10 veces con diferentes inicializaciones
        max_iter=300         # Máximo de iteraciones por ejecución
    ))
])
pipeline

Ajustamos el pipeline a los datos. Esto escala los datos y realiza el clustering K-Means.

In [ ]:
pipeline.fit(X)

,steps,"[('scaler', ...), ('kmeans', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,n_clusters,10
,init,'k-means++'
,n_init,10
,max_iter,300


### Estadísticas de Entrenamiento

Examinemos los resultados del clustering:
- **Inertia**: Suma de distancias al cuadrado de las muestras a su centro de cluster más cercano. Menor es generalmente mejor (clusters más compactos), pero la inertia disminuye a medida que aumenta K, por lo que debe interpretarse con cuidado.
- **Tamaño de Clusters**: Verificamos si los clusters están relativamente equilibrados o si algunos están vacíos/muy pequeños.

In [ ]:
# Acceder a los componentes entrenados
scaler = pipeline.named_steps['scaler']
kmeans = pipeline.named_steps['kmeans']

# Obtener las asignaciones de clusters
labels_pred = kmeans.labels_

print(f"Número de clusters: {kmeans.n_clusters}")
print(f"Número de iteraciones: {kmeans.n_iter_}")
print(f"Inertia final (suma de distancias cuadradas a los centros): {kmeans.inertia_:.2f}")

# Mostrar tamaños de clusters
unique_labels, counts = np.unique(labels_pred, return_counts=True)
for cluster_id, count in zip(unique_labels, counts):
    bar = '█' * (count // 10)
    print(f"Cluster {cluster_id}: {count:3d} muestras {bar}")

### Interpretación de los Resultados del Entrenamiento

**1. Iteraciones**: El algoritmo convergió en **32 iteraciones** (muy por debajo del límite predeterminado de 300).
   - **Criterio de Convergencia**: K-Means deja de iterar cuando los centros de cluster se mueven menos que un umbral específico (tolerancia `tol=1e-4` por defecto) entre pasos consecutivos.
   - **Significado**: Converger en 32 iteraciones significa que los centroides se estabilizaron rápidamente y encontraron un mínimo local sin necesidad de ser detenidos forzosamente por `max_iter`.

**2. Inertia (69,813.56)**: ¿Es esto bueno o malo?
Para entender este número, lo comparamos con la variación total del dataset:

*   **Varianza Total (Suma Total de Cuadrados)**: Como estandarizamos los datos, cada feature (píxel) tiene varianza $1$. Con $1,797$ muestras y $64$ features, la inertia máxima posible (si tuviéramos solo 1 cluster) sería $\approx 1,797 \times 64 = 115,008$.
*   **Inertia (Varianza Intra-Cluster)**: Esta es la "variación" o variación que queda *dentro* de los clusters. Un valor de $69,813$ significa que $\approx 60\%$ de la variación total sigue dentro de los clusters ($69,813 / 115,008 = 0.607$).
*   **Varianza Explicada**: La estructura de clustering explica el $\approx 40\%$ restante de la variación. Esto representa qué tan bien separados están los 10 grupos.

**Conclusión**: Recuperar el 40% de la estructura con solo 10 centros simples es razonable para dígitos escritos a mano de alta dimensión, donde gran parte de la variación proviene de estilos individuales de escritura (inclinación, grosor) más que de la identidad del dígito.

**3. Tamaño de Clusters**: Vemos que todos los clusters tienen un número significativo de muestras (ninguno está vacío). Esto sugiere que el algoritmo no quedó atrapado en un mínimo local trivial y encontró una partición equilibrada.

### 4.3. Visualización de los Centros de Cluster

Como estamos en un espacio de 64 dimensiones, podemos visualizar los **centros de cluster** (centroides) para entender qué representa cada cluster. 
Como usamos `StandardScaler`, los centroides están en el espacio escalado. Usamos `scaler.inverse_transform` para mapearlos de vuelta al espacio original de intensidad de píxel (0-16) y reformarlos en imágenes 8x8.

Idealmente, cada centroide debería parecerse a un dígito claro.

In [ ]:
# Visualizar centros de clusters como imágenes de dígitos
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Centros de Cluster de K-Means (Dígito Promedio en Cada Cluster)', fontsize=14, fontweight='bold')

# Obtener centros de cluster en el espacio original (transformación inversa desde el espacio escalado)
centers_scaled = kmeans.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)

for i, ax in enumerate(axes.flat):
    # Reformar el centro a imagen 8x8
    center_image = centers_original[i].reshape(8, 8)
    ax.imshow(center_image, cmap='gray')
    ax.set_title(f'Cluster {i}\n({counts[i]} muestras)', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5. Evaluar el Rendimiento del Clustering

Evaluar el clustering es complicado porque en escenarios verdaderamente no supervisados no tenemos etiquetas "correctas". Sin embargo, como tenemos las etiquetas reales de los dígitos, podemos usar tanto métricas **no supervisadas** como **supervisadas**.

### Métricas No Supervisadas (Sin usar etiquetas reales)

Estas métricas evalúan la calidad del cluster basándose únicamente en la geometría de los clusters:

**Silhouette Score** (-1 a 1, mayor es mejor):
- Mide qué tan similares son los puntos a su propio cluster versus otros clusters
- Fórmula: $s = \frac{b - a}{\max(a, b)}$ donde $a$ = distancia promedio al propio cluster, $b$ = distancia promedio al cluster más cercano

**Calinski-Harabasz Score** (mayor es mejor):
- Ratio de dispersión entre clusters respecto a la dispersión intra-cluster
- Mayor significa clusters más densos y bien separados

**Davies-Bouldin Score** (menor es mejor):
- Similitud promedio entre cada cluster y su cluster más similar
- Menor significa mejor separación

In [ ]:
# Obtener datos escalados para evaluación
X_scaled = scaler.transform(X)

# Calcular métricas no supervisadas
silhouette = silhouette_score(X_scaled, labels_pred)
calinski_harabasz = calinski_harabasz_score(X_scaled, labels_pred)
davies_bouldin = davies_bouldin_score(X_scaled, labels_pred)

print("\nEstas métricas evalúan la calidad del cluster sin usar etiquetas reales:\n")
print(f"Silhouette Score:        {silhouette:8.4f}  (Rango: [-1, 1], mayor es mejor)")
print(f"Calinski-Harabasz Score: {calinski_harabasz:8.2f}  (Mayor es mejor)")
print(f"Davies-Bouldin Score:    {davies_bouldin:8.4f}  (Menor es mejor)")

# Interpretar Silhouette Score
if silhouette > 0.5:
    interpretation = "Buena separación de clusters"
elif silhouette > 0.25:
    interpretation = "Estructura moderada de clusters"
else:
    interpretation = "Estructura débil de clusters"
print(f"\nInterpretación del Silhouette Score: {interpretation}")


### Métricas Supervisadas (Usando etiquetas reales para comparación)

**Importante**: En el clustering no supervisado del mundo real, no tenemos etiquetas reales y dependemos de métricas intrínsecas (como Silhouette). Aquí usamos métricas externas (ARI, NMI) con fines de **benchmarking**, ya que la verdad de referencia está disponible para este dataset.

Estas métricas miden la concordancia entre los clusters predichos y las etiquetas reales:

**Adjusted Rand Index** (-1 a 1, mayor es mejor):
- Mide la similitud entre dos agrupamientos, ajustado por azar
- 1 = coincidencia perfecta, 0 = aleatoria, < 0 = peor que aleatoria

**Normalized Mutual Information** (0 a 1, mayor es mejor):
- Mide la información mutua entre las etiquetas de cluster y las etiquetas reales

**Homogeneity, Completeness, V-Measure**:
- Homogeneity: Cada cluster contiene solo miembros de una sola clase
- Completeness: Todos los miembros de una clase están en el mismo cluster
- V-Measure: Media armónica de homogeneity y completeness

In [ ]:
# Calcular métricas supervisadas (comparando con etiquetas reales)
ari = adjusted_rand_score(y_true, labels_pred)
nmi = normalized_mutual_info_score(y_true, labels_pred)
homogeneity = homogeneity_score(y_true, labels_pred)
completeness = completeness_score(y_true, labels_pred)
v_measure = v_measure_score(y_true, labels_pred)

print(f"Adjusted Rand Index (ARI):      {ari:8.4f}  (Rango: [-1, 1], mayor es mejor)")
print(f"Normalized Mutual Info (NMI):   {nmi:8.4f}  (Rango: [0, 1], mayor es mejor)")
print(f"Homogeneity:                    {homogeneity:8.4f}  (Cada cluster = un dígito)")
print(f"Completeness:                   {completeness:8.4f}  (Cada dígito = un cluster)")
print(f"V-Measure:                      {v_measure:8.4f}  (Media armónica)")

Crear un mapeo de clusters a dígitos (¿qué dígito es más común en cada cluster?)

In [ ]:
print(f"✓ Los clusters tienen un {v_measure*100:.1f}% de concordancia con las etiquetas reales de dígitos")
print(f"✓ Silhouette Score promedio de {silhouette:.3f} indica clusters razonablemente separados")

In [ ]:
cluster_to_digit = {}
for cluster_id in range(10):
    # Obtener todas las etiquetas reales para este cluster
    mask = labels_pred == cluster_id
    true_labels_in_cluster = y_true[mask]
    
    # Encontrar el dígito más común
    unique, counts = np.unique(true_labels_in_cluster, return_counts=True)
    most_common_digit = unique[np.argmax(counts)]
    purity = counts[np.argmax(counts)] / len(true_labels_in_cluster) * 100
    
    cluster_to_digit[cluster_id] = most_common_digit
    
    print(f"Cluster {cluster_id} → Dígito {most_common_digit} (Pureza: {purity:.1f}%)")
    
    # Mostrar distribución de todos los dígitos en este cluster
    dist = " | ".join([f"{d}:{cnt}" for d, cnt in zip(unique, counts)])
    print(f"            Distribución: {dist}\n")

## 6. Visualizar los Resultados del Clustering

Como nuestros datos tienen 64 dimensiones, no podemos visualizarlos directamente. Usaremos **PCA** (Principal Component Analysis) para reducirlos a 2D para su visualización.

### ¿Qué es PCA?

PCA encuentra las direcciones (componentes principales) en las que los datos varían más. Al proyectar los datos en los primeros 2-3 componentes, podemos visualizar datos de alta dimensión preservando la mayor parte de la varianza.

**Importante**: Esto es solo para visualización. El clustering real se realizó en el espacio completo de 64 dimensiones.

### Visualización y Análisis

#### 1. Proyección PCA (Izquierda y Centro)
- Los puntos cercanos en 2D también estaban cerca en el espacio de 64D
- Se espera algo de superposición: perdemos información al proyectar de 64D → 2D
- Compara los clusters predichos vs. las etiquetas reales para ver la concordancia


In [ ]:
# Aplicar PCA para reducir a 2D para visualización
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"\nReducidas {X.shape[1]} dimensiones a 2 componentes principales")
print(f"Varianza explicada por PC1: {pca.explained_variance_ratio_[0]*100:.2f}%")
print(f"Varianza explicada por PC2: {pca.explained_variance_ratio_[1]*100:.2f}%")
print(f"Varianza total explicada: {sum(pca.explained_variance_ratio_)*100:.2f}%")

# Crear visualización de Clusters vs Etiquetas Reales
fig = plt.figure(figsize=(12, 6))

# Gráfico 1: Coloreado por cluster predicho
ax1 = fig.add_subplot(121)
scatter1 = ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_pred, 
                       cmap='tab10', alpha=0.6, s=30, edgecolors='k', linewidth=0.5)
ax1.set_xlabel('Primer Componente Principal', fontsize=11)
ax1.set_ylabel('Segundo Componente Principal', fontsize=11)
ax1.set_title('Clusters Encontrados por K-Means', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
cbar1 = plt.colorbar(scatter1, ax=ax1, ticks=range(10))
cbar1.set_label('ID de Cluster', fontsize=10)

# Gráfico 2: Coloreado por dígito real
ax2 = fig.add_subplot(122)
scatter2 = ax2.scatter(X_pca[:, 0], X_pca[:, 1], c=y_true, 
                       cmap='tab10', alpha=0.6, s=30, edgecolors='k', linewidth=0.5)
ax2.set_xlabel('Primer Componente Principal', fontsize=11)
ax2.set_ylabel('Segundo Componente Principal', fontsize=11)
ax2.set_title('Etiquetas Reales de Dígitos', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
cbar2 = plt.colorbar(scatter2, ax=ax2, ticks=range(10))
cbar2.set_label('Dígito', fontsize=10)

plt.tight_layout()
plt.show()


#### 2. Gráfico Silhouette (Derecha)
- Cada barra horizontal = una muestra, ordenada por coeficiente silhouette
- Barras anchas por encima de la línea roja = bien agrupadas
- Barras delgadas o negativas = posiblemente mal clasificadas
- Todos los clusters tienen Silhouette Score promedio positivo ✓

In [ ]:
# Gráfico Silhouette analysis
fig = plt.figure(figsize=(8, 6))
ax3 = fig.add_subplot(111)
silhouette_vals = silhouette_samples(X_scaled, labels_pred)
y_lower = 10

for i in range(10):
    cluster_silhouette_vals = silhouette_vals[labels_pred == i]
    cluster_silhouette_vals.sort()
    
    size_cluster_i = cluster_silhouette_vals.shape[0]
    y_upper = y_lower + size_cluster_i
    
    color = plt.cm.tab10(i / 10)
    ax3.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_silhouette_vals,
                      facecolor=color, edgecolor=color, alpha=0.7)
    
    ax3.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
    y_lower = y_upper + 10

ax3.set_xlabel('Coeficiente Silhouette', fontsize=11)
ax3.set_ylabel('Cluster', fontsize=11)
ax3.set_title('Gráfico Silhouette', fontsize=13, fontweight='bold')
ax3.axvline(x=silhouette, color="red", linestyle="--", linewidth=2, label=f'Media: {silhouette:.3f}')
ax3.set_yticks([])
ax3.legend()
ax3.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()


#### 3. Imágenes de Muestra
- Muestra imágenes reales de dígitos de cada cluster
- La mayoría de los clusters contienen predominantemente un tipo de dígito
- Cierta confusión entre dígitos de apariencia similar (ej. 3 y 8, 4 y 9)


In [ ]:
# Mostrar imágenes de muestra de cada cluster
fig, axes = plt.subplots(10, 10, figsize=(15, 15))
fig.suptitle('Imágenes de Muestra de Cada Cluster (10 por cluster)', fontsize=16, fontweight='bold')

for cluster_id in range(10):
    # Obtener índices de imágenes en este cluster
    cluster_indices = np.where(labels_pred == cluster_id)[0]
    
    # Tomar muestra de hasta 10 imágenes
    sample_indices = np.random.choice(cluster_indices, min(10, len(cluster_indices)), replace=False)
    
    for i, img_idx in enumerate(sample_indices):
        ax = axes[cluster_id, i]
        image = X[img_idx].reshape(8, 8)
        ax.imshow(image, cmap='gray')
        ax.axis('off')
        
        if i == 0:
            # Mapear cluster al dígito más común
            most_common = cluster_to_digit[cluster_id]
            ax.set_ylabel(f'Cluster {cluster_id}\n(→Dígito {most_common})', 
                         fontsize=10, fontweight='bold')
        
        # Agregar etiquetas de columna en la fila superior
        if cluster_id == 0:
            ax.set_title(f'#{i+1}', fontsize=9, pad=5)

plt.tight_layout()
plt.show()


## 7. Elegir el Número de Clusters (K)

En nuestro ejemplo, sabíamos que K=10 porque hay 10 dígitos. ¿Pero qué pasa si no lo supiéramos? Exploremos métodos para elegir K.

### El Elbow Method

Graficar la **inertia** (suma de distancias al cuadrado a los centros de cluster) para diferentes valores de K. Buscar un "codo" donde la tasa de disminución cambia bruscamente.

### Silhouette Method

Graficar el Silhouette Score para diferentes valores de K. Elegir el K que maximiza el Silhouette Score.

Probemos ambos métodos:

In [ ]:
# Probar diferentes números de clusters
K_range = range(2, 21)
inertias = []
silhouettes = []

print("=" * 80)
print("=" * 80)
print("\nEsto puede tardar un minuto...\n")

for k in K_range:
    # Crear pipeline con k clusters
    pipeline_k = Pipeline([
        ('scaler', StandardScaler()),
        ('kmeans', KMeans(n_clusters=k, random_state=42, n_init=10))
    ])
    
    # Ajustar y evaluar
    pipeline_k.fit(X)
    labels_k = pipeline_k.named_steps['kmeans'].labels_
    X_scaled_k = pipeline_k.named_steps['scaler'].transform(X)
    
    inertias.append(pipeline_k.named_steps['kmeans'].inertia_)
    silhouettes.append(silhouette_score(X_scaled_k, labels_k))
    
    print(f"K={k:2d} | Inertia: {inertias[-1]:10,.2f} | Silhouette: {silhouettes[-1]:.4f}")

print("\n✓ ¡Evaluación completa!")

# Visualizar resultados
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico Elbow Method
axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=10, color='r', linestyle='--', linewidth=2, label='K=10 (número real)')
axes[0].set_xlabel('Número de Clusters (K)', fontsize=12)
axes[0].set_ylabel('Inertia (Suma de Distancias al Cuadrado)', fontsize=12)
axes[0].set_title('Elbow Method', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Gráfico Silhouette
axes[1].plot(K_range, silhouettes, 'go-', linewidth=2, markersize=8)
axes[1].axvline(x=10, color='r', linestyle='--', linewidth=2, label='K=10 (número real)')
axes[1].axhline(y=silhouettes[K_range.index(10)], color='r', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Número de Clusters (K)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Method', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Marcar el Silhouette Score máximo
best_k = K_range[np.argmax(silhouettes)]
axes[1].plot(best_k, max(silhouettes), 'r*', markersize=20, label=f'Mejor K={best_k}')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("=" * 80)
print(f"\nMejor K según Silhouette Score: {best_k}")
print(f"Número real de dígitos: 10")
print(f"\nElbow Method:")
print("  Busca dónde la curva se 'dobla' - la tasa de disminución se desacelera")
print(f"  El codo aparece alrededor de K=8-10")
print(f"\nSilhouette Method:")
print(f"  Silhouette máximo en K={best_k}: {max(silhouettes):.4f}")
print(f"  Silhouette en K=10: {silhouettes[K_range.index(10)]:.4f}")

print("\n✓ ¡Ambos métodos sugieren K alrededor de 8-12, confirmando nuestra elección de K=10!")

## 8. Conclusiones Principales

### Lo que Aprendimos

1. **Aprendizaje No Supervisado**: K-Means descubrió agrupaciones de dígitos sin ver nunca las etiquetas, logrando una concordancia significativa con las clases reales.

2. **El Escalado de Features es Fundamental**: K-Means utiliza la distancia euclídea, por lo que las features deben estar en escalas comparables. StandardScaler garantiza una contribución igual de todas las features.

3. **Pipelines para Clustering**: Aunque el clustering es no supervisado, los pipelines siguen aportando beneficios: código limpio, sin fuga de datos en el preprocesamiento y experimentación sencilla.

4. **Múltiples Métricas de Evaluación**: Se usaron tanto métricas no supervisadas (Silhouette, Inertia) como supervisadas (ARI, NMI) para entender el rendimiento desde diferentes perspectivas.

5. **Elegir K es Difícil**: En escenarios reales sin etiquetas verdaderas, usa el Elbow Method, análisis del Silhouette Score y conocimiento del dominio para seleccionar K.

### Limitaciones de K-Means

1. **Debe especificarse K**: Es necesario conocer o estimar el número de clusters
2. **Asume clusters esféricos**: Tiene dificultades con formas alargadas o irregulares
3. **Sensible a la inicialización**: Diferentes arranques aleatorios pueden dar resultados distintos (mitigado con `n_init`)
4. **Sensible a valores atípicos**: Los outliers pueden distorsionar los centros de cluster
5. **Asume varianza igual**: Presupone que los clusters tienen tamaños y densidades similares

### Cuándo Usar K-Means

**Adecuado para:**
- Datasets grandes (algoritmo eficiente)
- Clusters aproximadamente esféricos y bien separados
- Cuando se tiene una estimación razonable de K
- Análisis exploratorio inicial

**Considerar alternativas para:**
- Número desconocido de clusters → Prueba DBSCAN o Clustering Jerárquico
- Clusters no esféricos → Prueba DBSCAN o Spectral Clustering
- Densidades diferentes de clusters → Prueba DBSCAN
- Sensibilidad a outliers → Prueba DBSCAN o variantes robustas

### Buenas Prácticas Demostradas

✓ Siempre escalar features antes de K-Means  
✓ Usar pipelines para garantizar un preprocesamiento consistente  
✓ Probar múltiples valores de K y comparar métricas  
✓ Visualizar los resultados (incluso datos de alta dimensión mediante PCA)  
✓ Usar múltiples métricas de evaluación  
✓ Examinar el contenido de los clusters para verificar agrupaciones significativas  
✓ Establecer random_state para reproducibilidad  
✓ Usar n_init > 1 para encontrar la mejor inicialización